In [77]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import brier_score_loss, make_scorer, log_loss
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import shap

In [2]:
df = pd.read_csv("../Data/cleanedHealthData.csv")

In [3]:
## Establish weights for target variable
healthy_weight = df['target'].value_counts(normalize=True)['healthy']
diseased_weight = df['target'].value_counts(normalize=True)['diseased']

In [4]:
## Separate all the columns
numerical_cols = df.select_dtypes(include=['number']).columns
# numerical_cols = [col for col in numerical_cols if col != 'target']

categorical_cols = list(df.select_dtypes(exclude=['number']).columns)
categorical_cols = [col for col in categorical_cols if col != 'target']


In [5]:
## Set target as 1|0

df['target'] = df['target'].replace({'healthy': 0, 'diseased': 1})

y = df['target']
X = df.drop(columns='target')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

C:\Users\admin\AppData\Local\Temp\ipykernel_14208\3429725909.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['target'] = df['target'].replace({'healthy': 0, 'diseased': 1})


In [ ]:
## Setup Numerical and Categorical Cols Transformers (KNNImputer, Scaler and OHE)

num_transformer = Pipeline([
    ('imputer', KNNImputer(n_neighbors=3)), ## Reduce to 3 to save computation
    ('scaler', StandardScaler()),
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('ohe', OneHotEncoder(drop = 'first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols)
    ]
)

## For Random Forest only 
imp_transformer = Pipeline([
    ('imputer', KNNImputer(n_neighbors=3))
])

rf_preprocesser = ColumnTransformer(
    transformers = [
        ('num', imp_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols)
    ] 
)

## Baseline Logistic Regression Model

In [7]:
## Logistic Regression pipeline (0.210771)

base_models = {
    'LogisticRegression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(random_state=42))
    ]),
}

results = []
for model, pipeline in tqdm(base_models.items(), desc="Training Models"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df

Training Models: 100%|██████████| 1/1 [02:59<00:00, 179.77s/it]


,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210771,0.612582


In [35]:
def get_feature_names(preprocessor, numeric_cols, categorical_cols):
    feature_names = []

    if 'num' in preprocessor.named_transformers_:
        feature_names.extend(numeric_cols)

    if 'cat' in preprocessor.named_transformers_:
        ohe = preprocessor.named_transformers_['cat'].named_steps['ohe']
        ohe_names = ohe.get_feature_names_out(categorical_cols)
        feature_names.extend(ohe_names)

    return feature_names

## Use LASSO, Random Forest, KMeans for Feature Selection

In [ ]:
## LASSO Feature Selection
fs_models = {
    'Lasso': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(penalty='l1', solver='liblinear', random_state=42))
    ])
}

# results = []
for model, pipeline in tqdm(fs_models.items(), desc="Lasso Feature Selection"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    ## Get feature names (OHE produced more)
    pre = pipeline.named_steps['preprocessor']
    feature_names = get_feature_names(pre, numerical_cols, categorical_cols)
    
    lasso_importance = pd.Series(np.abs(pipeline['model'].coef_).flatten(), index=feature_names)
    lasso_features = set(lasso_importance[lasso_importance > 0].index)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df


Lasso Feature Selection: 100%|██████████| 1/1 [03:08<00:00, 188.91s/it]


,Model,Brier Score Loss,Log Loss
0,Lasso,0.210761,0.612558


In [126]:
lasso_importance[lasso_features]

age                         0.009931
height                      0.021619
weight                      0.058255
bmi_corrected               0.055450
waist_size                  0.001901
                              ...   
caffeine_intake_Missing     0.005984
caffeine_intake_Moderate    0.020515
family_history_Yes          0.001143
pet_owner_Yes               0.010964
gene_marker_flag_Missing    0.012455
Length: 61, dtype: float64

In [127]:
print(f"Lasso Feature Selection:\n{lasso_features}")

Lasso Feature Selection:
['age', 'height', 'weight', 'bmi_corrected', 'waist_size', 'blood_pressure', 'heart_rate', 'cholesterol', 'glucose', 'insulin', 'sleep_hours', 'work_hours', 'physical_activity', 'daily_steps', 'calorie_intake', 'sugar_intake', 'water_intake', 'screen_time', 'stress_level', 'mental_health_score', 'income', 'meals_per_day', 'daily_supplement_dosage', 'gender_Male', 'sleep_quality_Good', 'sleep_quality_Poor', 'alcohol_consumption_Occasionally', 'alcohol_consumption_Regularly', 'smoking_level_Light', 'smoking_level_Non-smoker', 'mental_health_support_Yes', 'education_level_High School', 'education_level_Master', 'education_level_PhD', 'job_type_Labor', 'job_type_Office', 'job_type_Service', 'job_type_Tech', 'job_type_Unemployed', 'occupation_Doctor', 'occupation_Driver', 'occupation_Engineer', 'occupation_Farmer', 'occupation_Teacher', 'diet_type_Omnivore', 'diet_type_Vegan', 'diet_type_Vegetarian', 'exercise_type_Missing', 'exercise_type_Mixed', 'exercise_type_Str

In [ ]:
## Random Forest Feature Selection
fs_models = {
    'RandomForest': Pipeline([
        ('preprocessor', rf_preprocesser),  ## No StandardScaler for RF
        ('model', RandomForestClassifier(random_state=42))
    ])
}

results = []
for model, pipeline in tqdm(fs_models.items(), desc="Random Forest Feature Selection"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    ## Get feature names (OHE produced more)
    pre = pipeline.named_steps['preprocessor']
    feature_names = get_feature_names(pre, numerical_cols, categorical_cols)
    
    rf_importance = pd.Series(pipeline['model'].feature_importances_, index=feature_names)
    rf_features = set(rf_importance[rf_importance > rf_importance.mean()].index)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df

Random Forest Feature Selection: 100%|██████████| 1/1 [03:43<00:00, 223.61s/it]


,Model,Brier Score Loss,Log Loss
0,RandomForest,0.21366,0.619371


In [129]:
print(f"Random Forest Feature Selection:\n{rf_features}")

Random Forest Feature Selection:
['age', 'height', 'weight', 'bmi_corrected', 'waist_size', 'blood_pressure', 'heart_rate', 'cholesterol', 'glucose', 'insulin', 'sleep_hours', 'work_hours', 'physical_activity', 'daily_steps', 'calorie_intake', 'sugar_intake', 'water_intake', 'screen_time', 'stress_level', 'mental_health_score', 'income', 'daily_supplement_dosage']


In [ ]:
## KNN uses permutation importance (which requires testing features repeatedly to check if loss metric changes)



In [81]:
for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

In [ ]:
## XGB Feature Selection
fs_models = {
    'XGB': Pipeline([
        ('model', XGBClassifier(enable_categorical=True, random_state=42))   ## No scaling needed
    ])
}

results = []
for model, pipeline in tqdm(fs_models.items(), desc="XGB Feature Selection"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    explainer = shap.TreeExplainer(pipeline['model'])
    shap_values = explainer.shap_values(X_train)
   
    
    shap_importance = pd.Series(np.abs(shap_values).mean(0), index=X.columns)
    xgb_feature_names = set(shap_importance.nlargest(30).index)

    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

results_df = pd.DataFrame(results).sort_values(by='Log Loss')
results_df

XGB Feature Selection: 100%|██████████| 1/1 [00:06<00:00,  6.54s/it]


,Model,Brier Score Loss,Log Loss
0,XGB,0.218409,0.632276


In [130]:
print(f"XGB Feature Selection:\n{xgb_feature_names}")

XGB Feature Selection:
['income', 'sugar_intake', 'bmi_corrected', 'sleep_hours', 'blood_pressure', 'daily_steps', 'insulin', 'cholesterol', 'weight', 'heart_rate', 'height', 'work_hours', 'waist_size', 'water_intake', 'physical_activity', 'glucose', 'job_type', 'screen_time', 'calorie_intake', 'age', 'daily_supplement_dosage', 'occupation', 'mental_health_score', 'education_level', 'sleep_quality', 'exercise_type', 'stress_level', 'diet_type', 'healthcare_access', 'device_usage']


In [ ]:
## Filtered out 22 features (length of rf_features)

selected_features = (
    lasso_features & rf_features |
    lasso_features & xgb_feature_names |
    rf_features & xgb_feature_names
)


print(len(selected_features))
print("\nFINAL SELECTED FEATURES:\n", selected_features)

22

FINAL SELECTED FEATURES:
 {'sugar_intake', 'screen_time', 'waist_size', 'water_intake', 'income', 'stress_level', 'age', 'sleep_hours', 'insulin', 'cholesterol', 'daily_supplement_dosage', 'height', 'glucose', 'work_hours', 'bmi_corrected', 'blood_pressure', 'calorie_intake', 'weight', 'heart_rate', 'mental_health_score', 'physical_activity', 'daily_steps'}


## Baseline Pipeline of all Models on Full Data

In [139]:
models = {
    'LogisticRegression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('preprocessor', rf_preprocesser),
        ('model', RandomForestClassifier(random_state=42))
    ]),
    'GradientBoosting': Pipeline([
        ('preprocessor', preprocessor),
        ('model', GradientBoostingClassifier(random_state=42))
    ]),
    'DecisionTree': Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'KNN': Pipeline([
        ('preprocessor', preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'XGBoost': Pipeline([
        ('model', XGBClassifier(enable_categorical=True, random_state=42))
    ])
}

In [140]:
results = []

for model, pipeline in tqdm(models.items(), desc="Training Baseline Models Full Data"):
    pipeline.fit(X_train, y_train)
    y_pred_prob = pipeline.predict_proba(X_test)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

baseline_results_df = pd.DataFrame(results).sort_values(by='Log Loss')
# results_df.to_csv('../Data/preliminaryResults.csv', index=False)
# results_df

Training Baseline Models Full Data: 100%|██████████| 6/6 [17:08<00:00, 171.48s/it]


In [143]:
baseline_results_df

,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210771,0.612582
2,GradientBoosting,0.210801,0.612651
1,RandomForest,0.213660,0.619371
5,XGBoost,0.218409,0.632276
4,KNN,0.251964,2.432118
3,DecisionTree,0.433200,15.614111


## Use Logistic Regression, Random Forest, Boosting Models to Evaluate Selection

In [148]:
## Filter dataset for selected features
X_train_selected = X_train[list(selected_features)]
X_test_selected = X_test[list(selected_features)]

selected_categorical_cols = [feature for feature in categorical_cols if feature in selected_features]
selected_numerical_cols = [feature for feature in numerical_cols if feature in selected_features]

## Adjust Preprocessor and Pipeline based on limited features

sel_preprocessor = ColumnTransformer(
    transformers = [
        ('num', num_transformer, selected_numerical_cols),
        ('cat', cat_transformer, selected_categorical_cols)
    ]
)

sel_rf_preprocesser = ColumnTransformer(
    transformers = [
        ('num', imp_transformer, selected_numerical_cols),
        ('cat', cat_transformer, selected_categorical_cols)
    ] 
)

sel_models = {
    'LogisticRegression': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', LogisticRegression(random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('preprocessor', sel_rf_preprocesser),
        ('model', RandomForestClassifier(random_state=42))
    ]),
    'GradientBoosting': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', GradientBoostingClassifier(random_state=42))
    ]),
    'DecisionTree': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'KNN': Pipeline([
        ('preprocessor', sel_preprocessor),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ]),
    'XGBoost': Pipeline([
        ('model', XGBClassifier(enable_categorical=True, random_state=42))
    ])
}


In [149]:
results = []

for model, pipeline in tqdm(sel_models.items(), desc="Training Baseline Models Selected Data"):
    pipeline.fit(X_train_selected, y_train)
    y_pred_prob = pipeline.predict_proba(X_test_selected)[:, 1]   ## class 1 is diseased
    
    brier = brier_score_loss(y_test, y_pred_prob)
    logloss = log_loss(y_test, y_pred_prob)
    
    results.append({
        'Model': model,
        'Brier Score Loss': brier,
        'Log Loss': logloss
    })

selected_results_df = pd.DataFrame(results).sort_values(by='Log Loss')
# results_df.to_csv('../Data/preliminaryResults.csv', index=False)
# results_df

Training Baseline Models Selected Data: 100%|██████████| 6/6 [17:19<00:00, 173.24s/it]


In [150]:
baseline_results_df

,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210771,0.612582
2,GradientBoosting,0.210801,0.612651
1,RandomForest,0.213660,0.619371
5,XGBoost,0.218409,0.632276
4,KNN,0.251964,2.432118
3,DecisionTree,0.433200,15.614111


In [151]:
selected_results_df

,Model,Brier Score Loss,Log Loss
0,LogisticRegression,0.210607,0.612188
2,GradientBoosting,0.210727,0.612485
1,RandomForest,0.213666,0.619433
5,XGBoost,0.219153,0.633917
4,KNN,0.251318,2.494181
3,DecisionTree,0.424700,15.307740


## Hyperparameter Tuning

In [ ]:
## Set parameters for each model

param_grids = {
    "GradientBoosting": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_depth": [3, 4, 5],
        "model__subsample": [0.8, 1.0],
    },
    "LogisticRegression": {
        "model__C": np.logspace(-3, 3, 10),
        "model__solver": ["liblinear", "lbfgs"],
        "model__penalty": ["l2", "l1"],
    },
    "RandomForest": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
        "model__metric": ["euclidean", "manhattan"],
    },
    "DecisionTree": {
        "model__max_depth": [None, 5, 10, 20],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__criterion": ["gini", "entropy"],
    },
}